# 04. 감소 분해 — 덜 왔나 덜 썼나(방문/지출) · 누가 줄었나(외지인/거주민)

**먼저 `유형나누기.ipynb`를 실행**해서 `유형나누기_결과.xlsx`를 새로 만든 뒤 이 노트북을 실행하세요. 취약 후보·순위·시간대 유형은 그 파일에서 그대로 가져옵니다.

**③ 방문/지출 분해** (V4의 매출 S·건수 N·baseline B, 낮·저녁·하루 각각)
- 평소 건수 = baseline ÷ 평시 객단가(같은 업종·시간대·요일). 그러면 $\frac{S}{B}=\frac{N}{N_{평소}}\times\frac{객단가}{객단가_{평소}}$ 로 정확히 나뉨
- 건수 몫 = $\ln(N/N_{평소})\,/\,\ln(S/B)$ → **70% 이상 방문 감소형**, **30% 이하 지출 감소형**, 그 사이 **혼합형**(매출이 줄지 않았으면 `감소 아님`)
- **결과는 업종의 시간대 유형에 맞는 시간대로 봄**: 낮 집중 감소형·시간 이동형 → 낮, 저녁 집중 감소형·저녁 감소형 → 저녁, 나머지 → 하루
- **보조 결과(여름 객단가)**: 평시 날은 대부분 9월 중순~12월이라, 여름에 원래 객단가가 낮은 업종은 '지출 감소'가 크게 보일 수 있음. 평소 객단가를 **여름 평시 날(8월·9월 상순)**로만 계산한 결과를 함께 제시(주 결과는 바꾸지 않음, 표본이 적어 요일 구분 없음)

**② 외지인/거주민 분해** (카드 데이터2, 하루 단위)
- 평소 외지인 매출 = baseline × 평시 외지인 비중(같은 업종·요일), 실제 외지인 매출 = 실제 매출 × 그날 외지인 비중
- 기여율 = 외지인 감소분 ÷ 전체 감소분, 쏠림 = 기여율 ÷ 평시 외지인 비중 (1보다 크면 외지인이 더 많이 줄었음)
- **판정**: 이벤트의 외지인 비중이 평시 범위(평시 날짜 1,000번 추출, 5~95%)보다 **낮은** 이벤트가 절반 이상·최소 2번 → **외지인 주도**, **높은** 이벤트가 절반 이상·최소 2번 → **거주민 주도**, 그 밖 → **쏠림 없음**
- 외지인 = 가맹점과 다른 시·도 거주 고객. 거주지 `정보없음`은 계산에서 뺌. 강남 "거주민"은 서울 시민 전체라 강남 결과는 참고로만.

기준값(70%·30%, 반복 조건, 여름 = 8월·9월 상순)은 실행 전에 정한 값입니다.

In [ ]:
# 0. 설정
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/big_contest')
try:
    from google.colab import drive; drive.mount('/content/drive')
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
except ImportError:
    pass
import numpy as np, pandas as pd, warnings, matplotlib.pyplot as plt
try: import koreanize_matplotlib
except ImportError: print('※ 한글 글꼴 패키지가 없어 그래프 글자가 깨질 수 있습니다')
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

def find(pattern):
    hits = sorted(ROOT.rglob(pattern), key=lambda p: p.stat().st_mtime)
    assert hits, f'{pattern} 을(를) {ROOT} 아래에서 찾지 못했습니다'
    print('읽는 파일:', hits[-1]); return hits[-1]
V4_PATH = find('38_최종baseline_V4.csv')
TYPE_PATH = find('유형나누기_결과.xlsx')
DATA2_PATH = find('신한카드_빅콘테스트2026_데이터2_수정.txt')      # 이름이 다르면 직접 경로를 넣으세요
OUT = ROOT/'results'/'감소분해'; OUT.mkdir(parents=True, exist_ok=True)

SPECIAL2_DAYS = pd.DatetimeIndex(np.concatenate([pd.date_range(a, b) for a, b in
                 [('2025-08-14', '2025-08-18'), ('2025-10-02', '2025-10-10'), ('2025-12-24', '2025-12-26')]]))
GAP_MERGE = 0
N_DRAW, RANGE_Q, SEED = 1000, (5, 95), 42
REPEAT_MIN, REPEAT_SHARE = 2, 0.5
VISIT_HI, VISIT_LO = 0.70, 0.30
SPANS = {'낮': ['06_11', '12_17'], '저녁': ['18_23'], '하루': None}
SUMMER = lambda d: (d.dt.month == 8) | ((d.dt.month == 9) & (d.dt.day <= 15))   # 보조 결과용 여름 평시 날
HOME = {'강남': '서울', '춘천': '강원'}

COL = dict(cat=['#2a78d6', '#eb6834', '#1baf7a', '#eda100'], ink='#0b0b0b', ink2='#52514e', grid='#e4e3df')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': COL['ink2'],
                     'axes.labelcolor': COL['ink2'], 'xtick.color': COL['ink2'], 'ytick.color': COL['ink2'],
                     'axes.grid': True, 'grid.color': COL['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'axes.unicode_minus': False, 'figure.dpi': 110})

In [ ]:
# 1. V4·이벤트·평시 날 (유형나누기와 같은 규칙)
P = pd.read_csv(V4_PATH, encoding='utf-8-sig', parse_dates=['날짜'])
assert 'N' in P.columns, 'V4 CSV에 결제건수 N 열이 없습니다'
P['특수일2'] = P['날짜'].isin(SPECIAL2_DAYS); P['특보일'] = (P['폭염등급'] > 0) | (P['호우등급'] > 0)
P['요일'] = P['날짜'].dt.dayofweek
ALL_DATES = pd.date_range(P['날짜'].min(), P['날짜'].max())
dd = P.drop_duplicates(['대상', '날짜']).set_index(['대상', '날짜']).sort_index()

def runs(s):
    out, cur = [], []
    for d, v in s.items():
        if v: cur.append(d)
        elif cur: out.append(cur); cur = []
    if cur: out.append(cur)
    return out
EP = []
for reg in sorted(P['대상'].unique()):
    w = dd.loc[reg]
    for typ, col in [('폭염', '폭염등급'), ('호우', '호우등급')]:
        s = w[col].reindex(ALL_DATES).fillna(0) > 0; merged = []
        for r in runs(s):
            if merged and (r[0] - merged[-1][-1]).days - 1 <= GAP_MERGE: merged[-1] = list(pd.date_range(merged[-1][0], r[-1]))
            else: merged.append(list(r))
        for i, days in enumerate(merged, 1):
            EP.append(dict(대상=reg, 종류=typ, 에피소드=f'{typ}{i}', 날짜=days, 분석일=[d for d in days if d not in SPECIAL2_DAYS]))
EP = pd.DataFrame(EP)
SPAN = {r: set(sum(EP.loc[EP['대상'] == r, '날짜'].tolist(), [])) for r in EP['대상'].unique()}
CALM = {r: set(z.index.get_level_values(1)[((z['평시']) & ~z['특수일2']).to_numpy()]) - SPAN[r] for r, z in dd.groupby(level=0)}
AN = {(r, t): set(sum(z['분석일'].tolist(), [])) for (r, t), z in EP.groupby(['대상', '종류'])}
P['평시풀'] = [d in CALM.get(r, ()) for r, d in zip(P['대상'], P['날짜'])]

CAND = pd.read_excel(TYPE_PATH, sheet_name='취약후보_유형')
CAND['gkey'] = CAND['대상'] + '|' + CAND['업종']
print('이벤트 수:', EP.groupby(['대상', '종류']).size().to_dict())
print('취약 후보:', CAND.groupby(['대상', '종류']).size().to_dict())
nx = CAND[CAND['시간대유형'] == '계산 불가']
if not nx.empty or '비고' not in CAND:
    print('※ 주의: 유형나누기 결과가 수정 전 버전일 수 있습니다(시간대 유형 "계산 불가":', nx['업종'].tolist(), ') → 수정된 유형나누기.ipynb를 먼저 다시 실행하세요')

def span_of(tt):   # 업종의 시간대 유형에 맞춰 볼 시간대
    return '낮' if tt in ('낮 집중 감소형', '시간 이동형') else ('저녁' if '저녁' in str(tt) else '하루')
CAND['분해시간대'] = CAND['시간대유형'].map(span_of)

## ③ 방문/지출 분해

In [ ]:
# 2. 평소 객단가 → 평소 건수 → 방문/지출 (주 결과: 평시 날 전체, 보조: 여름 평시 날)
A = P[~P['휴무']].copy()
calm = A[A['평시풀'] & (A['N'] > 0)]
ratio = lambda z: z['S'].sum() / z['N'].sum()
p_slot_wd = calm.groupby(['gkey', 'slot', '요일']).apply(ratio).rename('객단가_평시')
p_slot = calm.groupby(['gkey', 'slot']).apply(ratio).rename('객단가_평시2')
p_summer = calm[SUMMER(calm['날짜'])].groupby(['gkey', 'slot']).apply(ratio).rename('객단가_여름')
A = A.join(p_slot_wd, on=['gkey', 'slot', '요일']).join(p_slot, on=['gkey', 'slot']).join(p_summer, on=['gkey', 'slot'])
A['객단가_평시'] = A['객단가_평시'].fillna(A['객단가_평시2'])     # 그 요일 평시 표본이 없으면 요일 구분 없이
A['N_평소'] = A['B'] / A['객단가_평시']
A['N_평소_여름'] = A['B'] / A['객단가_여름'].fillna(A['객단가_평시'])  # 여름 평시 표본이 없는 칸은 주 결과 값
sm = calm[SUMMER(calm['날짜'])]
print('여름 평시 날 수:', sm.groupby('대상')['날짜'].nunique().to_dict())

def visit_spend(z, ncol):
    z = z[np.isfinite(z[ncol])]
    S, B, N, Nb = z['S'].sum(), z['B'].sum(), z['N'].sum(), z[ncol].sum()
    if B <= 0 or Nb <= 0 or N <= 0: return dict(매출변화=np.nan, 건수변화=np.nan, 객단가변화=np.nan, 건수몫=np.nan, 유형='계산 불가')
    rs, rn = S / B, N / Nb; rp = rs / rn
    if rs >= 1: share, t = np.nan, '감소 아님'
    else:
        share = np.log(rn) / np.log(rs)
        t = '방문 감소형' if share >= VISIT_HI else ('지출 감소형' if share <= VISIT_LO else '혼합형')
    return dict(매출변화=rs - 1, 건수변화=rn - 1, 객단가변화=rp - 1, 건수몫=share, 유형=t)

def run_vs(ncol):
    rows = []
    for _, c in CAND.iterrows():
        z0 = A[(A['gkey'] == c['gkey']) & A['날짜'].isin(AN[(c['대상'], c['종류'])])]
        row = dict(대상=c['대상'], 종류=c['종류'], gkey=c['gkey'])
        for name, sl in SPANS.items():
            z = z0 if sl is None else z0[z0['slot'].isin(sl)]
            r = visit_spend(z, ncol) if len(z) else dict(유형='해당 시간대 없음')
            row.update({f'{name}_{k}': v for k, v in r.items()})
        rows.append(row)
    return pd.DataFrame(rows)
VS, VS_S = run_vs('N_평소'), run_vs('N_평소_여름')

# 업종마다 시간대 유형에 맞는 시간대의 결과를 한 열로
def pick(V, suffix):
    M = CAND[['대상', '종류', 'gkey', '분해시간대']].merge(V, on=['대상', '종류', 'gkey'])
    out = M[['대상', '종류', 'gkey']].copy()
    for k in ['유형', '건수변화', '객단가변화', '매출변화']:
        out[f'분해_{k}{suffix}'] = [r[f"{r['분해시간대']}_{k}"] if f"{r['분해시간대']}_{k}" in r else np.nan for _, r in M.iterrows()]
    return out
PK = pick(VS, '').merge(pick(VS_S, '_여름'), on=['대상', '종류', 'gkey'])
PK = CAND[['대상', '종류', 'gkey', '업종', '순위', '시간대유형', '분해시간대']].merge(PK, on=['대상', '종류', 'gkey'])

print('[주 결과 — 해당 시간대 기준 방문/지출 유형]'); print(pd.crosstab([PK['대상'], PK['종류']], PK['분해_유형']).to_string())
print('\n[보조 — 여름 객단가 기준]'); print(pd.crosstab([PK['대상'], PK['종류']], PK['분해_유형_여름']).to_string())
CH = PK[PK['분해_유형'] != PK['분해_유형_여름']].copy()
for c_ in ['분해_건수변화', '분해_객단가변화', '분해_건수변화_여름', '분해_객단가변화_여름']: CH[c_] = (CH[c_] * 100).round(1)
print('\n[여름 객단가로 보면 유형이 바뀌는 업종]  (변화 %)')
print('없음' if CH.empty else CH[['대상', '종류', '순위', '업종', '분해시간대', '분해_유형', '분해_건수변화', '분해_객단가변화',
                                   '분해_유형_여름', '분해_건수변화_여름', '분해_객단가변화_여름']].sort_values(['대상', '종류', '순위']).to_string(index=False))

## ② 외지인/거주민 분해

In [ ]:
# 3. 데이터2 읽기
def read_raw(path):
    for enc in ['cp949', 'utf-8-sig', 'utf-8']:
        try: return pd.read_csv(path, sep='\t' if str(path).endswith('.txt') else ',', encoding=enc, dtype=str, low_memory=False)
        except UnicodeDecodeError: continue
    raise ValueError('인코딩을 알 수 없습니다')
R = read_raw(DATA2_PATH); R.columns = [c.strip().upper() for c in R.columns]
print(R.shape, list(R.columns))
for c in ['MCT_SGG_CD', 'CLN_SGG_CD', 'SEX_CCD']:
    print(f'{c}:', R[c].value_counts(dropna=False).to_dict())
R = R[~R['SEX_CCD'].fillna('').str.contains('법인')]
R['TS_AT'] = pd.to_numeric(R['TS_AT'].str.replace(',', ''), errors='coerce').fillna(0)
R['날짜'] = pd.to_datetime(R['TA_YMD'].str.replace(r'\D', '', regex=True), format='%Y%m%d')
V4_REG = sorted(P['대상'].unique())
R['대상'] = R['MCT_SGG_CD'].map(lambda x: next((v for v in V4_REG if v[:2] in str(x)), None))
R = R[R['대상'].notna()]
R['gkey'] = R['대상'] + '|' + R['MCT_RY_CD'].str.strip()
cln = R['CLN_SGG_CD'].fillna('').str.strip()
known = (cln != '') & (cln != '정보없음')
R['외지인'] = [HOME[r[:2]] not in c for c, r in zip(cln, R['대상'])]
R2 = R[known]
G2 = pd.DataFrame({'외지인': R2[R2['외지인']].groupby(['gkey', '날짜'])['TS_AT'].sum(),
                   '거주지있음': R2.groupby(['gkey', '날짜'])['TS_AT'].sum()}).fillna(0).reset_index()
G2['비중'] = G2['외지인'] / G2['거주지있음'].replace(0, np.nan)
G2['대상'] = G2['gkey'].str.split('|').str[0]; G2['요일'] = G2['날짜'].dt.dayofweek
G2['평시풀'] = [d in CALM.get(r, ()) for r, d in zip(G2['대상'], G2['날짜'])]
print(f"거주지 정보없음 매출 비중 {R.loc[~known, 'TS_AT'].sum() / R['TS_AT'].sum():.4f}, 집계 {len(G2):,}행, 후보 중 데이터2에 없는 업종:",
      sorted(set(CAND['gkey']) - set(G2['gkey'])))

In [ ]:
# 4. 이벤트별 외지인 비중 판정 + 기여율·쏠림
rng = np.random.default_rng(SEED)
pool = G2[G2['평시풀'] & (G2['거주지있음'] > 0)]
WDP = {g: {w_: (q['외지인'].to_numpy(), q['거주지있음'].to_numpy()) for w_, q in z.groupby('요일')} for g, z in pool.groupby('gkey')}
SH_WD = pool.groupby(['gkey', '요일']).apply(lambda z: z['외지인'].sum() / z['거주지있음'].sum())
SH_ALL = pool.groupby('gkey').apply(lambda z: z['외지인'].sum() / z['거주지있음'].sum())
Dday = P[~P['휴무']].groupby(['gkey', '날짜'])[['S', 'B']].sum()

def share_range(g, dates):
    a = np.zeros(N_DRAW); b = np.zeros(N_DRAW)
    for d in dates:
        q = WDP.get(g, {}).get(d.dayofweek)
        if q is None: return np.nan, np.nan          # 그 요일 평시 표본이 없으면 판정하지 않음
        ix = rng.integers(0, len(q[0]), N_DRAW); a += q[0][ix]; b += q[1][ix]
    return tuple(np.percentile(a / b, RANGE_Q))

def decomp(g, dates):
    # 실제/평소 외지인 매출을 V4 매출·baseline 규모로 맞춤
    z = G2[(G2['gkey'] == g) & G2['날짜'].isin(dates)].set_index('날짜')
    out = dict(실제외=0., 평소외=0., 실제=0., 평소=0.)
    for d in dates:
        if (g, d) not in Dday.index or d not in z.index or not np.isfinite(z.loc[d, '비중']): continue
        S, B = Dday.loc[(g, d)]; sh_d = z.loc[d, '비중']
        sh_b = SH_WD.get((g, d.dayofweek), SH_ALL.get(g, np.nan))
        if not np.isfinite(sh_b): continue
        out['실제외'] += S * sh_d; out['평소외'] += B * sh_b; out['실제'] += S; out['평소'] += B
    if out['평소'] <= 0: return None
    dec_all = out['실제'] - out['평소']; dec_out = out['실제외'] - out['평소외']
    base_sh = out['평소외'] / out['평소']
    contrib = dec_out / dec_all if dec_all < 0 else np.nan
    return dict(평시외지인비중=base_sh, 기여율=contrib, 쏠림=contrib / base_sh if np.isfinite(contrib) and base_sh > 0 else np.nan,
                외지인변화=out['실제외'] / out['평소외'] - 1 if out['평소외'] > 0 else np.nan,
                거주민변화=(out['실제'] - out['실제외']) / (out['평소'] - out['평소외']) - 1 if out['평소'] > out['평소외'] else np.nan)

ev = []
for _, c in CAND.iterrows():
    for _, e in EP[(EP['대상'] == c['대상']) & (EP['종류'] == c['종류'])].iterrows():
        z = G2[(G2['gkey'] == c['gkey']) & G2['날짜'].isin(e['분석일']) & (G2['거주지있음'] > 0)]
        if z.empty: continue
        sh = z['외지인'].sum() / z['거주지있음'].sum(); lo, hi = share_range(c['gkey'], list(z['날짜']))
        j = '계산 불가' if not np.isfinite(lo) else ('외지인 더 감소' if sh < lo else '거주민 더 감소' if sh > hi else '차이 없음')
        ev.append(dict(대상=c['대상'], 종류=c['종류'], gkey=c['gkey'], 에피소드=e['에피소드'], 외지인비중=sh, 평시하한=lo, 평시상한=hi, 판정=j))
EVO = pd.DataFrame(ev)

rows = []
for _, c in CAND.iterrows():
    z = EVO[(EVO['gkey'] == c['gkey']) & (EVO['종류'] == c['종류'])]; ok = z['판정'] != '계산 불가'; n = int(ok.sum())
    k_out, k_res = int((z['판정'] == '외지인 더 감소').sum()), int((z['판정'] == '거주민 더 감소').sum())
    lab = ('외지인 주도' if k_out >= REPEAT_MIN and k_out / max(n, 1) >= REPEAT_SHARE else
           '거주민 주도' if k_res >= REPEAT_MIN and k_res / max(n, 1) >= REPEAT_SHARE else '쏠림 없음' if n else '계산 불가')
    d = decomp(c['gkey'], AN[(c['대상'], c['종류'])]) or {}
    rows.append(dict(대상=c['대상'], 종류=c['종류'], gkey=c['gkey'], 고객층=lab, 외지인더감소=f'{k_out}/{n}', 거주민더감소=f'{k_res}/{n}', **d))
OUTD = pd.DataFrame(rows)
print('[고객층 판정]'); print(pd.crosstab([OUTD['대상'], OUTD['종류']], OUTD['고객층']).to_string())

## 결과 표와 한 줄 요약
- `분해시간대`: 그 업종의 시간대 유형에 맞춰 본 시간대(낮/저녁/하루)
- `유형(여름)`: 평소 객단가를 여름 평시 날로 계산한 보조 결과. 주 결과와 다르면 '지출 감소'에 계절이 섞였을 가능성

In [ ]:
# 5. 합치기
T = CAND.merge(PK.drop(columns=['업종', '순위', '시간대유형', '분해시간대']), on=['대상', '종류', 'gkey'], how='left') \
        .merge(OUTD, on=['대상', '종류', 'gkey'], how='left')
pc = lambda x: '-' if not np.isfinite(x) else f'{x*100:+.0f}%'
def sentence(r):
    s = f"{r['업종']}: {r['시간대유형']}"
    part = r['분해시간대']
    if r['분해_유형'] in ('방문 감소형', '지출 감소형', '혼합형'):
        s += f". {part} 감소는 {r['분해_유형']}(건수 {pc(r['분해_건수변화'])}, 객단가 {pc(r['분해_객단가변화'])})"
        if r['분해_유형_여름'] != r['분해_유형']:
            s += f" — 여름 객단가로 보면 {r['분해_유형_여름']}(건수 {pc(r['분해_건수변화_여름'])}, 객단가 {pc(r['분해_객단가변화_여름'])})"
    elif r['분해_유형'] == '감소 아님':
        s += f". {part} 매출이 평소보다 줄지 않음({pc(r['분해_매출변화'])})"
    if r['고객층'] in ('외지인 주도', '거주민 주도'):
        cr = r.get('기여율', np.nan)
        if not np.isfinite(cr): extra = ''
        elif cr > 1: extra = f", 외지인 매출 {pc(r['외지인변화'])}·거주민 {pc(r['거주민변화'])}(외지인 감소가 전체 감소보다 큼, 평소 외지인 비중 {r['평시외지인비중']*100:.0f}%)"
        elif cr < 0: extra = f", 외지인 매출 {pc(r['외지인변화'])}·거주민 {pc(r['거주민변화'])}(감소는 모두 거주민에게서)"
        else: extra = f", 하루 감소분의 {cr*100:.0f}%가 외지인(평소 비중 {r['평시외지인비중']*100:.0f}%)"
        s += f". 고객층: {r['고객층']}{extra}"
    elif r['고객층'] == '쏠림 없음':
        s += '. 고객층 쏠림 없음'
    return s
T['요약'] = T.apply(sentence, axis=1)

SHOW = ['순위', '업종', '시간대유형', '분해시간대', '분해_유형', '분해_건수변화', '분해_객단가변화', '분해_유형_여름',
        '고객층', '외지인더감소', '거주민더감소', '평시외지인비중', '기여율', '쏠림', '외지인변화', '거주민변화']
V = T.sort_values(['대상', '종류', '순위']).copy()
for c in ['분해_건수변화', '분해_객단가변화', '분해_건수변화_여름', '분해_객단가변화_여름', '분해_매출변화',
          '평시외지인비중', '기여율', '외지인변화', '거주민변화']: V[c] = (V[c] * 100).round(1)
V['쏠림'] = V['쏠림'].round(2)
V = V.rename(columns={'분해_유형': '유형', '분해_건수변화': '건수변화', '분해_객단가변화': '객단가변화', '분해_유형_여름': '유형(여름)',
                      '분해_건수변화_여름': '건수변화(여름)', '분해_객단가변화_여름': '객단가변화(여름)', '분해_매출변화': '매출변화'})
SHOW = [{'분해_유형': '유형', '분해_건수변화': '건수변화', '분해_객단가변화': '객단가변화', '분해_유형_여름': '유형(여름)'}.get(c, c) for c in SHOW]
for (reg, typ), t in V.groupby(['대상', '종류']):
    print(f'\n■ {reg} {typ}  (변화·비중·기여율 %)'); print(t[SHOW].to_string(index=False))
    print('  요약:'); [print('   -', s) for s in t['요약']]
if any('강남' in r for r in V['대상']): print('\n※ 강남의 거주민은 서울 시민 전체라 강남 고객층 결과는 참고용입니다.')

KEEP = ['대상', '종류'] + SHOW + ['매출변화', '건수변화(여름)', '객단가변화(여름)', '업종성격', '피해모양', '요약']
with pd.ExcelWriter(OUT/'감소분해_결과.xlsx') as xw:
    V[[c for c in KEEP if c in V]].to_excel(xw, sheet_name='취약후보_분해', index=False)
    EVO.to_excel(xw, sheet_name='이벤트별_외지인판정', index=False)
    VS.to_excel(xw, sheet_name='방문지출_낮저녁하루', index=False)
    VS_S.to_excel(xw, sheet_name='방문지출_여름객단가', index=False)
print('\n저장:', OUT/'감소분해_결과.xlsx')

## 그래프
7. **방문/지출**: 지역·기상별 상위 후보의 건수 변화와 객단가 변화(업종마다 해당 시간대 기준, 막대 옆 표시는 시간대)
8. **외지인/거주민**: 지역·기상별 상위 후보의 외지인·거주민 매출 변화(하루)

In [ ]:
# 6. 그래프 7·8
def paired(ax, t, a, b, la, lb, title, tag=None):
    t = t.iloc[::-1]; y = np.arange(len(t)); h = 0.38
    ax.barh(y + h / 2, t[a] * 100, h, color=COL['cat'][0], label=la)
    ax.barh(y - h / 2, t[b] * 100, h, color=COL['cat'][1], label=lb)
    lab = [f"{int(r)}위 {u}" + (f" ({s})" if tag is not None else '') for r, u, s in zip(t['순위'], t['업종'], t[tag] if tag else t['업종'])]
    ax.set_yticks(y, lab, fontsize=8)
    ax.axvline(0, color=COL['ink2'], lw=1); ax.grid(axis='y', visible=False)
    ax.set_xlabel('평소 대비 변화(%)'); ax.set_title(title, fontsize=10, color=COL['ink'], loc='left')

TOPK = 8
groups = [(k, g.sort_values('순위').head(TOPK)) for k, g in T.groupby(['대상', '종류'])]
for a, b, la, lb, fname, ttl, tag in [
        ('분해_건수변화', '분해_객단가변화', '건수(손님 수)', '객단가', '그래프7_방문지출', '덜 왔나(건수) vs 덜 썼나(객단가) — 업종별 해당 시간대', '분해시간대'),
        ('외지인변화', '거주민변화', '외지인', '거주민', '그래프8_외지인거주민', '외지인 vs 거주민 매출 변화 — 하루', None)]:
    fig, axes = plt.subplots(1, len(groups), figsize=(4.6 * len(groups), 0.42 * TOPK + 1.8), squeeze=False)
    for ax, ((reg, typ), g) in zip(axes[0], groups):
        paired(ax, g, a, b, la, lb, f'{reg[:2]} {typ}', tag)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, frameon=False, fontsize=9, loc='upper right', ncol=2, bbox_to_anchor=(1, 1.06))
    fig.suptitle(ttl, fontsize=11, color=COL['ink'], x=0.01, ha='left', y=1.06)
    fig.tight_layout(); fig.savefig(OUT/f'{fname}.png', bbox_inches='tight'); plt.show()
print('그래프 저장 폴더:', OUT)